first best

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC, NuSVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

# 1. Chargement des données
df = pd.read_csv("parkinson.csv")
X = df.drop(columns=['ID', 'Recording', 'Status'])
y = df['Status']
X = pd.get_dummies(X, columns=['Gender'], drop_first=True)

# 2. Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Sélection de caractéristiques avec c-SVM comme base (First Best)
svm_base = SVC(kernel='rbf', C=1.0)
selector = SequentialFeatureSelector(
    estimator=svm_base,
    direction='forward',
    cv=5,
    n_jobs=-1
)
X_selected = selector.fit_transform(X_scaled, y)
selected_features = X.columns[selector.get_support()]
print(f"Nombre de caractéristiques sélectionnées : {len(selected_features)}")
print("Caractéristiques sélectionnées :", list(selected_features))

# 4. Classifieurs à évaluer
models = {
    'Naïve Bayes': GaussianNB(),
    'c-SVM': SVC(kernel='rbf'),
    'nu-SVM': NuSVC(nu=0.5, kernel='rbf'),
    'MLP': MLPClassifier(hidden_layer_sizes=(100,), max_iter=2000, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

# 5. Validation croisée 10-fold + Résultats détaillés
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
results = []

for name, model in models.items():
    print(f"\n Validation croisée pour le modèle : {name}")
    y_pred = cross_val_predict(model, X_selected, y, cv=kf)
    acc = accuracy_score(y, y_pred)
    prec = precision_score(y, y_pred)
    rec = recall_score(y, y_pred)
    f1 = f1_score(y, y_pred)
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    })

    # Affichage du rapport de classification
    print(classification_report(y, y_pred, digits=3))

    # Affichage de la matrice de confusion
    cm = confusion_matrix(y, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_ if hasattr(model, 'classes_') else [0, 1])
    disp.plot(cmap='Blues')
    plt.title(f"Matrice de Confusion - {name}")
    plt.grid(False)
    plt.show()

# 6. Tableau final
results_df = pd.DataFrame(results)
print("\nRésumé des performances (Validation croisée 10-fold):\n")
print(results_df)

# 7. Courbe de performance
results_df.set_index("Model")[["Accuracy", "Precision", "Recall", "F1-Score"]].plot(kind="bar", figsize=(12, 6))
plt.title("Performance des Classifieurs avec Wrapper (First Best, base = c-SVM)")
plt.ylabel("Score")
plt.ylim(0.6, 1)
plt.grid(axis='y')
plt.tight_layout()
plt.show()


greedy stepwise

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC, NuSVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

# 1. Chargement des données
df = pd.read_csv("parkinson.csv")
X = df.drop(columns=['ID', 'Recording', 'Status'])
y = df['Status']
X = pd.get_dummies(X, columns=['Gender'], drop_first=True)

# 2. Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Sélection de caractéristiques avec c-SVM comme base (First Best)
svm_base = SVC(kernel='rbf', C=1.0)
selector = SequentialFeatureSelector(
    estimator=svm_base,
    direction='backward',
    cv=5,
    n_jobs=-1
)
X_selected = selector.fit_transform(X_scaled, y)
selected_features = X.columns[selector.get_support()]
print(f"Nombre de caractéristiques sélectionnées : {len(selected_features)}")
print("Caractéristiques sélectionnées :", list(selected_features))

# 4. Classifieurs à évaluer
models = {
    'Naïve Bayes': GaussianNB(),
    'c-SVM': SVC(kernel='rbf'),
    'nu-SVM': NuSVC(nu=0.5, kernel='rbf'),
    'MLP': MLPClassifier(hidden_layer_sizes=(100,), max_iter=2000, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

# 5. Validation croisée 10-fold + Résultats détaillés
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
results = []

for name, model in models.items():
    print(f"\n🔍 Validation croisée pour le modèle : {name}")
    y_pred = cross_val_predict(model, X_selected, y, cv=kf)
    acc = accuracy_score(y, y_pred)
    prec = precision_score(y, y_pred)
    rec = recall_score(y, y_pred)
    f1 = f1_score(y, y_pred)
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    })

    # Affichage du rapport de classification
    print(classification_report(y, y_pred, digits=3))

    # Affichage de la matrice de confusion
    cm = confusion_matrix(y, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_ if hasattr(model, 'classes_') else [0, 1])
    disp.plot(cmap='Blues')
    plt.title(f"Matrice de Confusion - {name}")
    plt.grid(False)
    plt.show()

# 6. Tableau final
results_df = pd.DataFrame(results)
print("\nRésumé des performances (Validation croisée 10-fold):\n")
print(results_df)

# 7. Courbe de performance
results_df.set_index("Model")[["Accuracy", "Precision", "Recall", "F1-Score"]].plot(kind="bar", figsize=(12, 6))
plt.title("Performance des Classifieurs avec Wrapper (First Best, base = c-SVM)")
plt.ylabel("Score")
plt.ylim(0.6, 1)
plt.grid(axis='y')
plt.tight_layout()
plt.show()


PSO

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, NuSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import pyswarms as ps

# 1. Chargement des données
df = pd.read_csv("parkinson.csv")
X = df.drop(columns=['ID', 'Recording', 'Status'])
y = df['Status']
X = pd.get_dummies(X, columns=['Gender'], drop_first=True)

# 2. Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Fonction d’évaluation pour PSO avec c-SVM
def fitness_function(particles):
    scores = []
    for particle in particles:
        mask = particle > 0.5
        if np.sum(mask) == 0:
            scores.append(1.0)
            continue
        X_sub = X_scaled[:, mask]
        score = cross_val_score(SVC(kernel='rbf', C=1.0), X_sub, y, cv=5, scoring='accuracy').mean()
        scores.append(1 - score)  # minimisation
    return np.array(scores)

# 4. Paramètres PSO
n_particles = 20
dimensions = X_scaled.shape[1]
options = {'c1': 2, 'c2': 2, 'w': 0.9, 'k': 5, 'p': 2}

# 5. Optimisation PSO
optimizer = ps.discrete.BinaryPSO(n_particles=n_particles, dimensions=dimensions, options=options)
best_cost, best_pos = optimizer.optimize(fitness_function, iters=30, verbose=True)

# 6. Sélection des meilleures caractéristiques
selected_mask = best_pos > 0.5
X_selected = X_scaled[:, selected_mask]
selected_features = X.columns[selected_mask]
print(f"\nNombre de caractéristiques sélectionnées : {np.sum(selected_mask)}")
print("Caractéristiques sélectionnées :", list(selected_features))

# 7. Modèles à évaluer
models = {
    'Naïve Bayes': GaussianNB(),
    'c-SVM': SVC(kernel='rbf'),
    'nu-SVM': NuSVC(nu=0.5, kernel='rbf'),
    'MLP': MLPClassifier(hidden_layer_sizes=(100,), max_iter=2000, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

# 8. Évaluation croisée 10-fold
results = []
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

for name, model in models.items():
    print(f"\n Validation croisée pour le modèle : {name}")
    y_pred = cross_val_predict(model, X_selected, y, cv=kf)
    acc = accuracy_score(y, y_pred)
    prec = precision_score(y, y_pred)
    rec = recall_score(y, y_pred)
    f1 = f1_score(y, y_pred)
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    })

    print(classification_report(y, y_pred, digits=3))
    cm = confusion_matrix(y, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
    disp.plot(cmap='Blues')
    plt.title(f"Matrice de Confusion - {name}")
    plt.grid(False)
    plt.show()

# 9. Tableau des performances
results_df = pd.DataFrame(results)
print("\nRésumé des performances (PSO avec c-SVM pour sélection des features):\n")
print(results_df)

# 10. Graphe comparatif
results_df.set_index("Model")[["Accuracy", "Precision", "Recall", "F1-Score"]].plot(kind="bar", figsize=(12, 6))
plt.title("Performance des Classifieurs avec Wrapper PSO (base = c-SVM)")
plt.ylabel("Score")
plt.ylim(0.6, 1)
plt.grid(axis='y')
plt.tight_layout()
plt.show()
